# Porto and Lisbon Urban Heat Island Exposure and Population Characteristics

This notebook reads the prepared real-data table for **Lisbon** and **Porto** and reproduces the core exposure and representation metrics used throughout this project.

The question is narrow by design:

> Which groups are more or less represented in cells with stronger modelled urban heat island intensity?


## Sources and interpretation

This analysis combines three official or near-official inputs:

- the **Eurostat GISCO Census 2021 1 km grid** for total population, children, older residents, employed residents, and people born outside the EU;
- the **Eurostat GISCO Urban Audit `CITIES` polygons** for Lisbon and Porto;
- the **EEA public service exposing the Copernicus/UrbClim urban heat island model**.

Two interpretation points matter before reading the results:

1. The Urban Audit `CITIES` polygons are **not municipality boundaries**, so the city totals here are larger than the municipality-only populations many readers expect.
2. Some coastal or no-data fragments do not return a direct UHI pixel value from the public service. Those fragments were filled with the nearest non-missing value within the same city during the preparation step, and that assumption is documented in `data/SOURCES.md`.


## Method

This notebook works from the prepared table rather than rebuilding the geospatial pipeline inline.

At a high level, the preparation workflow does this:

1. Intersect the Eurostat 1 km grid with the Lisbon and Porto Urban Audit city polygons.
2. Apply area weights when a grid cell is split by a city boundary.
3. Attach one UHI intensity value to each resulting fragment.
4. Mark fragments as exposed when `uhi_intensity_celsius >= 2.0` by default.
5. Compare each group's share in the whole city with its share inside exposed fragments.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd

from uhi_exposure.io import load_exposure_cells, write_dataframe
from uhi_exposure.metrics import compute_city_exposure_summary, compute_group_representation
from uhi_exposure.plotting import (
    plot_exposed_population_share,
    plot_group_share_comparison,
    plot_representation_ratios,
)

## Configuration

The notebook is configured for the real prepared table by default.

Change the threshold only if you want to test a different exposure definition. Keep in mind that a higher threshold can materially reduce the exposed population share, especially in Lisbon with the current source combination.


In [2]:
USE_DEMO = False
UHI_THRESHOLD_CELSIUS = 2.0
REAL_DATA_PATH = PROJECT_ROOT / "data" / "manual" / "porto_lisbon_cells.csv"
DEMO_DATA_PATH = PROJECT_ROOT / "data" / "demo" / "porto_lisbon_demo_cells.csv"
INPUT_PATH = DEMO_DATA_PATH if USE_DEMO else REAL_DATA_PATH
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PLOT_DIR = PROJECT_ROOT / "plots"

print(f"Using input: {INPUT_PATH}")
print(f"UHI threshold: {UHI_THRESHOLD_CELSIUS:.1f} °C")


Using input: C:\Users\diogo\work_code\ds-projects-portfolio\projects\porto_lisbon_uhi_exposure\data\manual\porto_lisbon_cells.csv
UHI threshold: 2.0 °C


## Load the prepared cell-level table

This cell reads the prepared CSV and shows the first rows so you can verify the schema and the expected fractional counts from area weighting.

Expected columns:

```text
city,cell_id,population_total,age_0_14,age_65_plus,employed,born_outside_eu,uhi_intensity_celsius,is_uhi_exposed
```


In [3]:
cells = load_exposure_cells(INPUT_PATH)
cells.head()

,city,cell_id,population_total,age_0_14,age_65_plus,employed,born_outside_eu,uhi_intensity_celsius,is_uhi_exposed
0,Lisbon,CRS3035RES1000mN1929000E2657000,0.000000,0.000000,0.000000,0.000000,0.000000,0.140198,False
1,Lisbon,CRS3035RES1000mN1933000E2674000,0.658670,0.099566,0.107225,0.245087,0.045954,1.645599,False
2,Lisbon,CRS3035RES1000mN1930000E2674000,1.879807,0.244013,0.478989,0.759153,0.126525,1.669006,False
3,Lisbon,CRS3035RES1000mN1929000E2674000,3.050808,0.435830,0.174332,1.220323,0.087166,1.477478,False
4,Lisbon,CRS3035RES1000mN1925000E2668000,29.370680,3.012377,6.150271,14.434308,4.769598,1.265320,False


In [4]:
cells.groupby("city", as_index=False).agg(
    cells=("cell_id", "count"),
    total_population=("population_total", "sum"),
    mean_uhi=("uhi_intensity_celsius", "mean"),
    max_uhi=("uhi_intensity_celsius", "max"),
)

,city,cells,total_population,mean_uhi,max_uhi
0,Lisbon,793,1.850999e+06,0.714196,2.234589
1,Porto,585,9.583587e+05,1.110140,2.300751


## City-level exposure summary

This table answers a first-order question: what share of each city's population lives in fragments above the selected UHI threshold?

If the Lisbon exposed share looks unexpectedly low, read it together with the source notes: the result reflects the combination of the Urban Audit city polygon, the published UHI raster footprint, and the 2.0 °C threshold.


In [5]:
city_summary = compute_city_exposure_summary(cells, threshold=UHI_THRESHOLD_CELSIUS)
city_summary

,city,total_population,exposed_population,exposed_population_share,mean_uhi_intensity_celsius,mean_uhi_intensity_exposed_celsius
0,Lisbon,1.850999e+06,19518.178566,0.010545,1.030735,2.121989
1,Porto,9.583587e+05,61532.000000,0.064206,1.507904,2.095122


## Group representation in exposed areas

The next table compares each group's share in the full city population with its share inside exposed fragments.

- A **representation ratio above 1** means the group is overrepresented in exposed areas.
- A **representation ratio below 1** means the group is underrepresented in exposed areas.


In [6]:
representation = compute_group_representation(cells, threshold=UHI_THRESHOLD_CELSIUS)
representation.sort_values(["city", "representation_ratio"], ascending=[True, False])

,city,group,city_group_count,exposed_group_count,city_group_share,exposed_group_share,difference_percentage_points,representation_ratio,is_overrepresented
3,Lisbon,born_outside_eu,2.842576e+05,4794.454585,0.153570,0.245640,9.207067,1.599536,True
2,Lisbon,not_employed,1.049505e+06,11677.865529,0.566994,0.598307,3.131362,1.055227,True
0,Lisbon,children_0_14,2.592130e+05,2860.390920,0.140039,0.146550,0.651063,1.046491,True
1,Lisbon,older_65_plus,4.211565e+05,4052.274713,0.227529,0.207615,-1.991385,0.912478,False
7,Porto,born_outside_eu,6.081594e+04,4453.000000,0.063458,0.072369,0.891042,1.140413,True
5,Porto,older_65_plus,2.161447e+05,15677.000000,0.225536,0.254778,2.924172,1.129654,True
6,Porto,not_employed,5.506193e+05,38353.000000,0.574544,0.623302,4.875764,1.084863,True
4,Porto,children_0_14,1.189934e+05,6832.000000,0.124164,0.111032,-1.313204,0.894236,False


## Compact interpretation table

This view converts the core representation metrics into a smaller table that is easier to use in writeups, slide decks, or social posts.


In [7]:
interpretation = representation.assign(
    city_group_share_pct=lambda df: 100 * df["city_group_share"],
    exposed_group_share_pct=lambda df: 100 * df["exposed_group_share"],
).loc[
    :,
    [
        "city",
        "group",
        "city_group_share_pct",
        "exposed_group_share_pct",
        "difference_percentage_points",
        "representation_ratio",
        "is_overrepresented",
    ],
]

interpretation.round(2)

,city,group,city_group_share_pct,exposed_group_share_pct,difference_percentage_points,representation_ratio,is_overrepresented
0,Lisbon,children_0_14,14.00,14.66,0.65,1.05,True
1,Lisbon,older_65_plus,22.75,20.76,-1.99,0.91,False
2,Lisbon,not_employed,56.70,59.83,3.13,1.06,True
3,Lisbon,born_outside_eu,15.36,24.56,9.21,1.60,True
4,Porto,children_0_14,12.42,11.10,-1.31,0.89,False
5,Porto,older_65_plus,22.55,25.48,2.92,1.13,True
6,Porto,not_employed,57.45,62.33,4.88,1.08,True
7,Porto,born_outside_eu,6.35,7.24,0.89,1.14,True


## Save outputs

This cell writes the city summary and the group representation table to `outputs/` so the numeric results can be reused outside the notebook.


In [8]:
summary_path = OUTPUT_DIR / ("demo_city_summary.csv" if USE_DEMO else "porto_lisbon_city_summary.csv")
representation_path = OUTPUT_DIR / ("demo_group_representation.csv" if USE_DEMO else "porto_lisbon_group_representation.csv")

write_dataframe(city_summary, summary_path)
write_dataframe(representation, representation_path)

summary_path, representation_path

(WindowsPath('C:/Users/diogo/work_code/ds-projects-portfolio/projects/porto_lisbon_uhi_exposure/outputs/porto_lisbon_city_summary.csv'),
 WindowsPath('C:/Users/diogo/work_code/ds-projects-portfolio/projects/porto_lisbon_uhi_exposure/outputs/porto_lisbon_group_representation.csv'))

## Plots

These charts are intentionally simple. At this stage the goal is to verify that the real pipeline produces coherent metrics and readable comparisons, not to optimize visual styling.


In [9]:
plot_exposed_population_share(
    city_summary,
    PLOT_DIR / ("demo_exposed_population_share.png" if USE_DEMO else "porto_lisbon_exposed_population_share.png"),
)
plot_representation_ratios(
    representation,
    PLOT_DIR / ("demo_representation_ratios.png" if USE_DEMO else "porto_lisbon_representation_ratios.png"),
)

for city in sorted(representation["city"].unique()):
    safe_city = city.lower().replace(" ", "_")
    plot_group_share_comparison(
        representation,
        city=city,
        output_path=PLOT_DIR / (f"demo_{safe_city}_group_share_comparison.png" if USE_DEMO else f"{safe_city}_group_share_comparison.png"),
    )

sorted(PLOT_DIR.glob("demo_*.png" if USE_DEMO else "*.png"))

[WindowsPath('C:/Users/diogo/work_code/ds-projects-portfolio/projects/porto_lisbon_uhi_exposure/plots/demo_exposed_population_share.png'),
 WindowsPath('C:/Users/diogo/work_code/ds-projects-portfolio/projects/porto_lisbon_uhi_exposure/plots/demo_lisbon_group_share_comparison.png'),
 WindowsPath('C:/Users/diogo/work_code/ds-projects-portfolio/projects/porto_lisbon_uhi_exposure/plots/demo_porto_group_share_comparison.png'),
 WindowsPath('C:/Users/diogo/work_code/ds-projects-portfolio/projects/porto_lisbon_uhi_exposure/plots/demo_representation_ratios.png'),
 WindowsPath('C:/Users/diogo/work_code/ds-projects-portfolio/projects/porto_lisbon_uhi_exposure/plots/exposed_population_share.png'),
 WindowsPath('C:/Users/diogo/work_code/ds-projects-portfolio/projects/porto_lisbon_uhi_exposure/plots/lisbon_group_share_comparison.png'),
 WindowsPath('C:/Users/diogo/work_code/ds-projects-portfolio/projects/porto_lisbon_uhi_exposure/plots/porto_group_share_comparison.png'),
 WindowsPath('C:/Users/diog

## Real-data rebuild notes

The geospatial preparation is implemented in `scripts/prepare_real_data.py`.

Run it when you want to refresh `data/manual/porto_lisbon_cells.csv` from source data instead of reusing the prepared table:

```bash
poetry run python scripts/prepare_real_data.py --output data/manual/porto_lisbon_cells.csv --threshold 2.0
```


## Writing guidance

When interpreting the outputs, keep four caveats explicit:

1. The cities are **Urban Audit cities**, not municipalities.
2. The UHI values come from a **published modelled raster**, not local station observations.
3. The exposed share depends directly on the **threshold choice**.
4. This is a **spatial representation** analysis, not a causal health-effects estimate.


## Caveats

- The threshold is a modelling decision, not a natural law.
- The output inherits the strengths and limits of the Eurostat grid and the published UrbClim service.
- Area-weighted fragments create fractional counts by construction.
- Results can differ sharply from municipality-based expectations because the Urban Audit city polygons are larger than municipal boundaries.
